# Company research: permitted-source data workflow

Use the instructor workbook, Capital IQ workbook and `Research_Data.csv.gz` from the latest Release. All restricted academic-database data and dependent rankings are excluded. No credentials are stored here. Other providers require their own authorized access; local access does not imply public redistribution permission.


In [ ]:
from pathlib import Path
import csv,json,gzip,os
import pandas as pd
csv.field_size_limit(2**31-1)
DATA=Path("Research_Data.csv.gz")
assert DATA.exists(), "Download the current Release CSV beside this notebook."
def records(dataset=None):
    with gzip.open(DATA,"rt",encoding="utf-8-sig",newline="") as f:
        for row in csv.DictReader(f):
            if dataset is None or row["dataset"]==dataset:
                yield row
def table(dataset):
    return pd.DataFrame(json.loads(r["record_json"]) for r in records(dataset))
from collections import Counter
print(Counter(r["dataset"] for r in records()))


## View research, sources and permitted target inputs
Profile rows preserve exact evidence text and source links. A missing value remains unknown. General and industry updates are separate record types linked by company ID. Scores withdrawn after the source change are not silently reused.


In [ ]:
profiles=table("Unified_Public_Research")
display(profiles[[c for c in ["key","name","main_filled","industry_filled"] if c in profiles]].head())
inputs=table("Restricted_Target_Inputs")
display(inputs.head())
if len(profiles):
    profile=profiles.iloc[0].to_dict()
    display(pd.DataFrame(profile.get("sources",{})).T)
    display(pd.DataFrame(profile.get("evidence",{})).T)


## Original workbooks
Only the two permitted supplied workbooks are inputs. Retain file, sheet and cell provenance for imported observations.


In [ ]:
for name in ["Company_List_for_Interns.xlsx","CapitalIQ_560.xlsx"]:
    p=Path(name)
    if p.exists():
        print(name,pd.ExcelFile(p).sheet_names)


## Source retrieval examples
SEC supplies reported facts, not consensus estimates. LSEG refresh requires an authorized local session and separately entitled fields. Target SEC data is research-only; it is never an input to simulated-private matching or forecasts. Functions below do not run automatically.


In [ ]:
import requests
def fetch_sec_facts(cik):
    contact=os.environ.get("SEC_USER_AGENT")
    if not contact: raise RuntimeError("Configure your contact SEC_USER_AGENT locally.")
    url=f"https://data.sec.gov/api/xbrl/companyfacts/CIK{int(cik):010d}.json"
    response=requests.get(url,headers={"User-Agent":contact},timeout=60)
    response.raise_for_status()
    return {"source_url":url,"record":response.json()}
def fetch_lseg(instruments,fields):
    import lseg.data as ld
    ld.open_session()
    try: return ld.get_data(universe=instruments,fields=fields)
    finally: ld.close_session()


## Rebuild and validate publication data
`code/exclude_restricted_sources.py` filters source/evidence graphs, invalidates inherited claims, withdraws stale rankings and validates all compressed website payloads. `code/build_unified_research_site.py` refuses a contaminated snapshot. The five-year forecast design is in `code/04_Revenue_Growth_Methodology.md`; company forecasts are not yet executed.
